# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [ ]:
# Load the AviationData.csv dataset
aviation_df = pd.read_csv("data/AviationData.csv", encoding='cp1252', on_bad_lines='skip')
aviation_df

/var/folders/xj/3fqt44g11p73jkrkp149njbm0000gn/T/ipykernel_12979/3884093065.py:2: DtypeWarning: Columns (6,7,28) have mixed types. Specify dtype option on import or set low_memory=False.
  aviation_df = pd.read_csv("AviationData.csv", encoding='cp1252', on_bad_lines='skip')


,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
0,20001218X45444,Accident,SEA87LA080,1948-10-24,"MOOSE CREEK, ID",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,UNK,Cruise,Probable Cause,NaN
1,20001218X45447,Accident,LAX94LA336,1962-07-19,"BRIDGEPORT, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,4.0,0.0,0.0,0.0,UNK,Unknown,Probable Cause,19-09-1996
2,20061025X01555,Accident,NYC07LA005,1974-08-30,"Saltville, VA",United States,36.922223,-81.878056,NaN,NaN,...,Personal,NaN,3.0,NaN,NaN,NaN,IMC,Cruise,Probable Cause,26-02-2007
3,20001218X45448,Accident,LAX96LA321,1977-06-19,"EUREKA, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,IMC,Cruise,Probable Cause,12-09-2000
4,20041105X01764,Accident,CHI79FA064,1979-08-02,"Canton, OH",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,1.0,2.0,NaN,0.0,VMC,Approach,Probable Cause,16-04-1980
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
88884,20221227106491,Accident,ERA23LA093,2022-12-26,"Annapolis, MD",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,0.0,1.0,0.0,0.0,NaN,NaN,NaN,29-12-2022
88885,20221227106494,Accident,ERA23LA095,2022-12-26,"Hampton, NH",United States,NaN,NaN,NaN,NaN,...,NaN,NaN,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN
88886,20221227106497,Accident,WPR23LA075,2022-12-26,"Payson, AZ",United States,341525N,1112021W,PAN,PAYSON,...,Personal,NaN,0.0,0.0,0.0,1.0,VMC,NaN,NaN,27-12-2022
88887,20221227106498,Accident,WPR23LA076,2022-12-26,"Morgan, UT",United States,NaN,NaN,NaN,NaN,...,Personal,MC CESSNA 210N LLC,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN


## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [20]:
# Make a working copy
df = aviation_df.copy()

# Convert Event.Date to datetime, then create Event.Year
df["Event.Date"] = pd.to_datetime(df["Event.Date"], errors="coerce")
df["Event.Year"] = df["Event.Date"].dt.year

# Keep rows from 1983 onward
df = df[df["Event.Year"] >= 1983]

# Keep professional builds only (Amateur.Built = No)
df = df[df["Amateur.Built"].astype(str).str.upper().str.strip() == "NO"]

# Keep accidents only
df = df[df["Investigation.Type"].astype(str).str.upper().str.strip() == "ACCIDENT"]

# Keep airplane category only
df = df[df["Aircraft.Category"].astype(str).str.upper().str.strip() == "AIRPLANE"]

display(df[["Event.Date", "Make", "Model", "Amateur.Built", "Investigation.Type", "Aircraft.Category"]].head())

,Event.Date,Make,Model,Amateur.Built,Investigation.Type,Aircraft.Category
4171,1983-03-20,Piper,PA-28-140,No,Accident,Airplane
4285,1983-04-02,De Havilland,DHC-6,No,Accident,Airplane
5960,1983-08-21,Lockheed,"LEARSTAR, L-18-56",No,Accident,Airplane
6669,1983-10-28,Short Brothers,SD3-30,No,Accident,Airplane
6806,1983-11-13,Beech,C35,No,Accident,Airplane


### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [21]:
# Injury columns we need
injury_cols = [
    "Total.Fatal.Injuries",
    "Total.Serious.Injuries",
    "Total.Minor.Injuries",
    "Total.Uninjured"
]

# Convert to numeric and fill missing values with 0
for col in injury_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

# Total serious + fatal injuries
df["serious_fatal_injuries"] = df["Total.Fatal.Injuries"] + df["Total.Serious.Injuries"]

# Estimate total people onboard from all injury outcomes
df["total_people_onboard_est"] = (
    df["Total.Fatal.Injuries"]
    + df["Total.Serious.Injuries"]
    + df["Total.Minor.Injuries"]
    + df["Total.Uninjured"]
)

# Avoid division by zero by replacing 0 with NaN
df["total_people_onboard_est"] = df["total_people_onboard_est"].replace(0, np.nan)

# Serious/fatal injury rate
df["serious_fatal_rate"] = df["serious_fatal_injuries"] / df["total_people_onboard_est"]

# Keep values in valid range 0 to 1
df["serious_fatal_rate"] = df["serious_fatal_rate"].clip(0, 1)

display(df[["serious_fatal_injuries", "total_people_onboard_est", "serious_fatal_rate"]].head())

,serious_fatal_injuries,total_people_onboard_est,serious_fatal_rate
4171,2.0,2.0,1.000000
4285,1.0,5.0,0.200000
5960,13.0,26.0,0.500000
6669,1.0,30.0,0.033333
6806,0.0,1.0,0.000000


**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [22]:
# Clean Aircraft.damage text
df["Aircraft.damage"] = df["Aircraft.damage"].astype(str).str.strip().str.title()

# Replace obvious missing-like values with Unknown
df["Aircraft.damage"] = df["Aircraft.damage"].replace({"Nan": "Unknown", "": "Unknown"})

# Create destroyed column as binary
# 1 = Destroyed, 0 = Not destroyed
df["was_destroyed"] = (df["Aircraft.damage"] == "Destroyed").astype(int)

display(df["Aircraft.damage"].value_counts(dropna=False))

Aircraft.damage
Substantial    16965
Destroyed       2316
Unknown          509
Minor            147
Name: count, dtype: int64

### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [23]:
# Basic Make cleaning
df["Make"] = df["Make"].astype(str).str.strip().str.upper()

# Mark unknown/blank make values as missing
df["Make"] = df["Make"].replace({"": np.nan, "UNKNOWN": np.nan, "UNK": np.nan, "NAN": np.nan})

# Remove rows where Make is missing
df = df.dropna(subset=["Make"])

# Keep only makes with at least 50 records
make_counts = df["Make"].value_counts()
valid_makes = make_counts[make_counts >= 50].index
df = df[df["Make"].isin(valid_makes)]

display(df["Make"].value_counts().head(20))

Make
CESSNA                7029
PIPER                 3938
BEECH                 1392
BOEING                 553
MOONEY                 358
BELLANCA               217
AIR TRACTOR INC        217
MAULE                  215
CIRRUS DESIGN CORP     209
AIR TRACTOR            206
AERONCA                200
CHAMPION               157
GRUMMAN                146
LUSCOMBE               141
CIRRUS                 130
STINSON                129
NORTH AMERICAN         104
EMBRAER                 98
DEHAVILLAND             93
TAYLORCRAFT             93
Name: count, dtype: int64

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [24]:
# Clean Model column
df["Model"] = df["Model"].astype(str).str.strip().str.upper()

# Treat blank/unknown labels as missing
df["Model"] = df["Model"].replace({"": np.nan, "UNKNOWN": np.nan, "UNK": np.nan, "NAN": np.nan})

# Drop missing models
df = df.dropna(subset=["Model"])

# Check if a model appears under more than one make
model_make_counts = df.groupby("Model")["Make"].nunique()

# Create unique identifier for plane type
df["Make_Model"] = df["Make"] + " | " + df["Model"]

display(df[["Make", "Model", "Make_Model"]].head())

,Make,Model,Make_Model
4171,PIPER,PA-28-140,PIPER | PA-28-140
4285,DE HAVILLAND,DHC-6,DE HAVILLAND | DHC-6
6806,BEECH,C35,BEECH | C35
7084,CESSNA,180K,CESSNA | 180K
7708,BEECH,99,BEECH | 99


### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [25]:
# Clean selected categorical columns
cat_cols = ["Engine.Type", "Weather.Condition", "Purpose.of.flight", "Broad.phase.of.flight"]

for col in cat_cols:
    df[col] = df[col].astype(str).str.strip().str.upper()
    df[col] = df[col].replace({"": np.nan, "UNKNOWN": np.nan, "UNK": np.nan, "NAN": np.nan})

# Convert number of engines to numeric
df["Number.of.Engines"] = pd.to_numeric(df["Number.of.Engines"])

display(df["Number.of.Engines"].describe())

count    15115.000000
mean         1.135495
std          0.361626
min          0.000000
25%          1.000000
50%          1.000000
75%          1.000000
max          4.000000
Name: Number.of.Engines, dtype: float64

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [26]:
# Find columns with too many missing values
missing_percent = df.isna().mean() * 100

# drop columns with more than 80% missing values
cols_to_drop = missing_percent[missing_percent > 80].index.tolist()

# Create cleaned dataframe
cleaned_df = df.drop(columns=cols_to_drop).copy()

### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [27]:
# Save cleaned dataframe so it can be used in the analysis notebook
cleaned_df.to_csv("data/AviationData_cleaned.csv", index=False)